In [1]:
from pathlib import Path
import random

In [2]:
DATASET_DIR = Path(r"D:\Workspace\Repository\thesis\research\object-detection-engine\data\PlantDoc\txt") 
DATASET_DIR

WindowsPath('D:/Workspace/Repository/thesis/research/object-detection-engine/data/PlantDoc/txt')

In [7]:
train_images_dir = DATASET_DIR / "train" / "images"
train_labels_dir = DATASET_DIR / "train" / "labels"

valid_images_dir = DATASET_DIR / "valid" / "images"
valid_labels_dir = DATASET_DIR / "valid" / "labels"

valid_images_dir.mkdir(parents=True, exist_ok=True)
valid_labels_dir.mkdir(parents=True, exist_ok=True)

# 2) collect train images
image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
train_images = [p for p in train_images_dir.glob("*") if p.suffix.lower() in image_exts]

n_train = len(train_images)
if n_train == 0:
    raise RuntimeError("No images found in train/images")

val_count = max(1, int(0.10 * n_train))  # 10% of train
print(f"Found {n_train} train images; moving {val_count} to validation.")

random.seed(42)  # for reproducibility
val_images = random.sample(train_images, val_count)

for img_path in val_images:
    # corresponding label file
    label_path = train_labels_dir / (img_path.stem + ".txt")

    # move image
    img_dest = valid_images_dir / img_path.name
    img_path.rename(img_dest)

    # move label if it exists
    if label_path.exists():
        label_dest = valid_labels_dir / label_path.name
        label_path.rename(label_dest)
    else:
        print(f"Warning: no label found for {img_path.name}")

print("Done splitting: train/valid/test now exist.")


Found 2328 train images; moving 232 to validation.
Done splitting: train/valid/test now exist.


In [ ]:
splits = ["train", "valid", "test"]  

for split in splits:
    labels_dir = DATASET_DIR / split / "labels"
    if not labels_dir.exists():
        continue

    print(f"Processing labels in: {labels_dir}")
    for label_file in labels_dir.glob("*.txt"):
        lines_out = []
        with open(label_file, "r") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split()
                # force every class to 0 (single "unhealthy" class)
                parts[0] = "0"
                lines_out.append(" ".join(parts))

        # overwrite file
        with open(label_file, "w") as f:
            if lines_out:
                f.write("\n".join(lines_out) + "\n")
            else:
                f.write("")  # keep empty file if needed

print("Done: all labels now use class 0 only.")


Processing labels in: D:\Workspace\Repository\thesis\research\object-detection-engine\data\PlantDoc\txt\train\labels
Processing labels in: D:\Workspace\Repository\thesis\research\object-detection-engine\data\PlantDoc\txt\valid\labels
Processing labels in: D:\Workspace\Repository\thesis\research\object-detection-engine\data\PlantDoc\txt\test\labels
Done: all labels now use class 0 only.


In [1]:
import json
import random
from pathlib import Path
import shutil

COCO_DIR = Path(r"D:\Workspace\Repository\thesis\research\object-detection-engine\data\PlantDoc\coco")

train_dir = COCO_DIR / "train"
valid_dir = COCO_DIR / "valid"
valid_dir.mkdir(exist_ok=True)

train_json_path = train_dir / "_annotations.coco.json"

with open(train_json_path, "r") as f:
    coco_data = json.load(f)

images = coco_data["images"]
annotations = coco_data["annotations"]
categories = coco_data["categories"]

n_train = len(images)
val_count = max(1, int(0.10 * n_train))
print(f"Found {n_train} train images; splitting {val_count} to validation.")

random.seed(42)
val_images = random.sample(images, val_count)
val_image_ids = {img["id"] for img in val_images}

train_images = [img for img in images if img["id"] not in val_image_ids]
train_annotations = [ann for ann in annotations if ann["image_id"] not in val_image_ids]
val_annotations = [ann for ann in annotations if ann["image_id"] in val_image_ids]

train_coco = {
    "images": train_images,
    "annotations": train_annotations,
    "categories": categories
}

val_coco = {
    "images": val_images,
    "annotations": val_annotations,
    "categories": categories
}

with open(train_json_path, "w") as f:
    json.dump(train_coco, f, indent=2)

valid_json_path = valid_dir / "_annotations.coco.json"
with open(valid_json_path, "w") as f:
    json.dump(val_coco, f, indent=2)

for img_info in val_images:
    src = train_dir / img_info["file_name"]
    dst = valid_dir / img_info["file_name"]
    if src.exists():
        shutil.move(str(src), str(dst))

print("Done: COCO train/valid split complete.")

Found 2328 train images; splitting 232 to validation.
Done: COCO train/valid split complete.


In [2]:
import json
from pathlib import Path

COCO_DIR = Path(r"D:\Workspace\Repository\thesis\research\object-detection-engine\data\PlantDoc\coco")

splits = ["train", "valid", "test"]

for split in splits:
    json_path = COCO_DIR / split / "_annotations.coco.json"
    if not json_path.exists():
        continue

    print(f"Processing: {json_path}")
    
    with open(json_path, "r") as f:
        coco_data = json.load(f)

    coco_data["categories"] = [{"id": 0, "name": "leaf"}]

    for ann in coco_data["annotations"]:
        ann["category_id"] = 0

    with open(json_path, "w") as f:
        json.dump(coco_data, f, indent=2)

print("Done: all COCO annotations now use single 'leaf' class.")

Processing: D:\Workspace\Repository\thesis\research\object-detection-engine\data\PlantDoc\coco\train\_annotations.coco.json
Processing: D:\Workspace\Repository\thesis\research\object-detection-engine\data\PlantDoc\coco\valid\_annotations.coco.json
Processing: D:\Workspace\Repository\thesis\research\object-detection-engine\data\PlantDoc\coco\test\_annotations.coco.json
Done: all COCO annotations now use single 'leaf' class.
